# Mediterranean Ops Fortress — Pipeline Data Report

**Project:** Mediterranean Ops Fortress  
**Stack:** Open-Meteo + OpenAQ v3 → Delta Lake (Backblaze B2) → Gold marts + Anomaly Detection  
**Observability:** Grafana Cloud (`mohamedwillforge.grafana.net`)  

This notebook reads directly from the live Backblaze B2 Gold layer and documents what the pipeline has produced: geographic and temporal coverage, pollutant concentration levels, WHO guideline exceedance rates, wildfire risk distribution, and anomaly detection results.

All data is real — no synthetic values, no mock rows.

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from deltalake import DeltaTable
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# Load B2 credentials from .env (MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY)
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))
load_dotenv()  # fallback: current dir .env

# Style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f8f8",
    "axes.grid": True,
    "grid.color": "white",
    "grid.linewidth": 1.2,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = ["#2196F3", "#FF5722", "#4CAF50", "#FFC107", "#9C27B0", "#00BCD4"]
print("Environment loaded.")

In [ ]:
def storage_options() -> dict:
    endpoint = os.environ["MINIO_ENDPOINT"]
    return {
        "AWS_ENDPOINT_URL": endpoint,
        "AWS_ACCESS_KEY_ID": os.environ["MINIO_ACCESS_KEY"],
        "AWS_SECRET_ACCESS_KEY": os.environ["MINIO_SECRET_KEY"],
        "AWS_REGION": os.environ.get("AWS_REGION", "eu-central-003"),
        "AWS_ALLOW_HTTP": "true" if endpoint.startswith("http://") else "false",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }

GOLD = os.environ.get("MINIO_BUCKET_GOLD", "med-ops-mohamed-gold")
opts = storage_options()

def read_gold(table: str) -> pd.DataFrame:
    path = f"s3://{GOLD}/{table}"
    try:
        df = DeltaTable(path, storage_options=opts).to_pandas()
        print(f"  {table}: {len(df):,} rows")
        return df
    except Exception as e:
        print(f"  {table}: FAILED — {e}")
        return pd.DataFrame()

print("Loading Gold tables from B2...")
summary   = read_gold("daily_country_summary")
risk      = read_gold("wildfire_risk_index")
anomalies = read_gold("anomaly_alerts")
print("Done.")

---
## 1. Pipeline Coverage

How much data was collected, across which countries and dates.

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────────
dates      = sorted(summary["partition_date"].unique())
countries  = sorted(summary["country_code"].dropna().unique())
sources    = sorted(summary["source"].dropna().unique())

print(f"Date range  : {dates[0]}  →  {dates[-1]}")
print(f"Dates total : {len(dates)}")
print(f"Countries   : {len(countries)}  —  {', '.join(countries)}")
print(f"Sources     : {', '.join(sources)}")
print(f"\nSummary rows    : {len(summary):,}")
print(f"Risk index rows : {len(risk):,}")
print(f"Anomaly rows    : {len(anomalies):,}")
total_stations = risk["station_id"].nunique() if not risk.empty else "N/A"
print(f"Unique stations : {total_stations}")

In [ ]:
# ── Station-day heatmap per country ────────────────────────────────────────────
pivot = (
    summary
    .groupby(["partition_date", "country_code"])["station_count"]
    .sum()
    .unstack("country_code", fill_value=0)
)
pivot.index = pd.to_datetime(pivot.index)

fig, ax = plt.subplots(figsize=(14, max(4, len(pivot.columns) * 0.45)))
im = ax.imshow(
    pivot.values.T,
    aspect="auto",
    cmap="YlOrRd",
    interpolation="nearest",
)
ax.set_yticks(range(len(pivot.columns)))
ax.set_yticklabels(pivot.columns, fontsize=9)
ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels(
    [d.strftime("%b %d") for d in pivot.index],
    rotation=60, ha="right", fontsize=7
)
plt.colorbar(im, ax=ax, label="Station-days")
ax.set_title("Station Coverage per Country × Date", fontsize=13, fontweight="bold", pad=12)
ax.set_facecolor("white")
plt.tight_layout()
plt.savefig("coverage_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Pollutant Concentrations by Country

Mean PM2.5 and O3 across all collected dates, split by data source.

In [ ]:
# ── Mean PM2.5 by country × source ────────────────────────────────────────────
pm25_by_country = (
    summary
    .groupby(["country_code", "source"])["mean_pm2_5"]
    .mean()
    .round(2)
    .unstack("source", fill_value=np.nan)
    .sort_values(by=summary["source"].mode()[0], ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PM2.5
ax = axes[0]
x = np.arange(len(pm25_by_country))
w = 0.35
for i, (src, col) in enumerate(zip(pm25_by_country.columns, PALETTE)):
    vals = pm25_by_country[src].values
    bars = ax.barh(x + i * w - w/2, vals, height=w, label=src, color=col, alpha=0.85)
ax.axvline(15, color="red", linestyle="--", linewidth=1, label="WHO PM2.5 guideline (15 µg/m³)")
ax.set_yticks(x)
ax.set_yticklabels(pm25_by_country.index)
ax.set_xlabel("Mean PM2.5 (µg/m³)")
ax.set_title("Mean PM2.5 by Country", fontweight="bold")
ax.legend(fontsize=8)

# O3
o3_by_country = (
    summary
    .groupby(["country_code", "source"])["mean_o3"]
    .mean()
    .round(2)
    .unstack("source", fill_value=np.nan)
    .reindex(pm25_by_country.index)
)
ax = axes[1]
for i, (src, col) in enumerate(zip(o3_by_country.columns, PALETTE)):
    vals = o3_by_country[src].values
    ax.barh(x + i * w - w/2, vals, height=w, label=src, color=col, alpha=0.85)
ax.axvline(100, color="red", linestyle="--", linewidth=1, label="WHO O3 guideline (100 µg/m³)")
ax.set_yticks(x)
ax.set_yticklabels(pm25_by_country.index)
ax.set_xlabel("Mean O3 (µg/m³)")
ax.set_title("Mean O3 by Country", fontweight="bold")
ax.legend(fontsize=8)

plt.suptitle("Air Pollutant Concentrations — All Dates Combined", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("pollutants_by_country.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. WHO Guideline Exceedance

Percentage of station-days where concentrations exceeded WHO 2021 guidelines.

In [ ]:
who_cols = {
    "who_pm25_exceed_pct": "PM2.5",
    "who_pm10_exceed_pct": "PM10",
    "who_no2_exceed_pct":  "NO2",
    "who_o3_exceed_pct":   "O3",
}
who_cols = {k: v for k, v in who_cols.items() if k in summary.columns}

exceed = (
    summary
    .groupby("country_code")[list(who_cols.keys())]
    .mean()
    .round(1)
    .rename(columns=who_cols)
)
exceed = exceed.sort_values(by=list(who_cols.values())[0], ascending=True)

fig, ax = plt.subplots(figsize=(11, max(4, len(exceed) * 0.55)))
x = np.arange(len(exceed))
n = len(who_cols)
w = 0.18
for i, (col, color) in enumerate(zip(exceed.columns, PALETTE)):
    ax.barh(
        x + (i - n / 2 + 0.5) * w,
        exceed[col],
        height=w,
        label=col,
        color=color,
        alpha=0.85,
    )
ax.set_yticks(x)
ax.set_yticklabels(exceed.index)
ax.set_xlabel("% of station-days exceeding WHO guideline")
ax.set_title("WHO Guideline Exceedance Rate by Country", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title="Pollutant", loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("who_exceedance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nCountries with highest PM2.5 exceedance rate:")
print(exceed[["PM2.5"]].sort_values("PM2.5", ascending=False).head(5).to_string())

---
## 4. Wildfire Risk Index

Composite score (0–100) per station × day: 60% normalised O3 + 40% normalised PM2.5.  
Levels: **low** (<25) · **moderate** (25–50) · **high** (50–75) · **extreme** (>75)

In [ ]:
if risk.empty:
    print("No wildfire risk data available.")
else:
    level_order = ["low", "moderate", "high", "extreme"]
    level_colors = {"low": "#4CAF50", "moderate": "#FFC107", "high": "#FF5722", "extreme": "#B71C1C"}

    # ── Distribution of risk levels by country ─────────────────────────────────
    risk_counts = (
        risk
        .groupby(["country_code", "risk_level"])
        .size()
        .unstack("risk_level", fill_value=0)
        .reindex(columns=[l for l in level_order if l in risk["risk_level"].unique()], fill_value=0)
    )
    risk_pct = risk_counts.div(risk_counts.sum(axis=1), axis=0) * 100
    risk_pct = risk_pct.sort_values(by=[c for c in ["extreme", "high"] if c in risk_pct.columns], ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(risk_pct) * 0.55)))

    # Stacked bar: risk level distribution
    ax = axes[0]
    left = np.zeros(len(risk_pct))
    for level in risk_pct.columns:
        vals = risk_pct[level].values
        ax.barh(risk_pct.index, vals, left=left, label=level, color=level_colors.get(level, "grey"), alpha=0.9)
        left += vals
    ax.set_xlabel("% of station-days")
    ax.set_title("Risk Level Distribution by Country", fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend(title="Risk level", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

    # Scatter: O3 vs PM2.5 coloured by risk level
    ax = axes[1]
    for level in level_order:
        mask = risk["risk_level"] == level
        if mask.any():
            ax.scatter(
                risk.loc[mask, "pm2_5"],
                risk.loc[mask, "ozone"],
                c=level_colors[level],
                label=level,
                alpha=0.5,
                s=12,
            )
    ax.set_xlabel("PM2.5 (µg/m³)")
    ax.set_ylabel("O3 (µg/m³)")
    ax.set_title("PM2.5 vs O3 — Risk Level", fontweight="bold")
    ax.legend(title="Risk level", fontsize=8)

    plt.suptitle("Wildfire Risk Index", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("wildfire_risk.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nTop 10 highest-risk station-days:")
    cols = [c for c in ["partition_date", "country_code", "station_name", "risk_index", "risk_level", "pm2_5", "ozone"] if c in risk.columns]
    print(risk.nlargest(10, "risk_index")[cols].to_string(index=False))

---
## 5. Anomaly Detection

Isolation Forest trained on the full Silver history (PM2.5, O3, NO2).  
Contamination rate: 5% — ~51 anomalies flagged from 1,016 rows.

In [ ]:
if anomalies.empty:
    print("No anomaly data available.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # ── Anomalies per date ─────────────────────────────────────────────────────
    ax = axes[0]
    anom_by_date = (
        anomalies
        .groupby("partition_date")["is_anomaly"]
        .sum()
        .sort_index()
    )
    ax.bar(range(len(anom_by_date)), anom_by_date.values, color="#FF5722", alpha=0.8)
    ax.set_xticks(range(len(anom_by_date)))
    ax.set_xticklabels(
        [d[-5:] for d in anom_by_date.index],  # MM-DD
        rotation=70, ha="right", fontsize=7
    )
    ax.set_ylabel("Anomalies flagged")
    ax.set_title("Anomalies per Date", fontweight="bold")

    # ── Anomaly rate by country ────────────────────────────────────────────────
    ax = axes[1]
    anom_by_country = (
        anomalies
        .groupby("country_code")["is_anomaly"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "anomalies", "count": "total"})
    )
    anom_by_country["rate"] = (anom_by_country["anomalies"] / anom_by_country["total"] * 100).round(1)
    anom_by_country = anom_by_country.sort_values("rate", ascending=True)
    ax.barh(anom_by_country.index, anom_by_country["rate"], color="#9C27B0", alpha=0.8)
    ax.set_xlabel("Anomaly rate (%)")
    ax.set_title("Anomaly Rate by Country", fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())

    # ── Anomaly score distribution ─────────────────────────────────────────────
    ax = axes[2]
    normal = anomalies.loc[anomalies["is_anomaly"] == 0, "anomaly_score"]
    flagged = anomalies.loc[anomalies["is_anomaly"] == 1, "anomaly_score"]
    ax.hist(normal, bins=30, color="#4CAF50", alpha=0.7, label=f"Normal ({len(normal):,})")
    ax.hist(flagged, bins=15, color="#FF5722", alpha=0.85, label=f"Anomaly ({len(flagged):,})")
    ax.set_xlabel("Isolation Forest score (lower = more anomalous)")
    ax.set_ylabel("Count")
    ax.set_title("Anomaly Score Distribution", fontweight="bold")
    ax.legend(fontsize=8)

    plt.suptitle("Anomaly Detection — Isolation Forest Results", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("anomaly_detection.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nTop 10 most anomalous readings (lowest score = most deviant):")
    cols = [c for c in ["partition_date", "country_code", "station_name", "pm2_5", "ozone", "nitrogen_dioxide", "anomaly_score"] if c in anomalies.columns]
    print(anomalies[anomalies["is_anomaly"] == 1].nsmallest(10, "anomaly_score")[cols].to_string(index=False))

---
## 6. Pipeline Summary

What the pipeline has produced as of this report.

In [ ]:
n_anomalies = int(anomalies["is_anomaly"].sum()) if not anomalies.empty else 0
n_anom_rows = len(anomalies)
anom_rate = f"{n_anomalies / n_anom_rows:.1%}" if n_anom_rows else "N/A"
top_risk_country = (
    risk.groupby("country_code")["risk_index"].mean().idxmax()
    if not risk.empty else "N/A"
)
highest_exceed_country = (
    summary.groupby("country_code")["who_pm25_exceed_pct"].mean().idxmax()
    if "who_pm25_exceed_pct" in summary.columns else "N/A"
)

report = pd.DataFrame([
    ["Date range",             f"{dates[0]} → {dates[-1]}"],
    ["Total dates collected",  len(dates)],
    ["Countries covered",      f"{len(countries)}  ({', '.join(countries)})"],
    ["Data sources",           ", ".join(sources)],
    ["Unique stations",        risk["station_id"].nunique() if not risk.empty else "N/A"],
    ["Gold summary rows",      f"{len(summary):,}"],
    ["Gold risk index rows",   f"{len(risk):,}"],
    ["Gold anomaly rows",      f"{n_anom_rows:,}"],
    ["Anomalies flagged",      f"{n_anomalies} ({anom_rate})"],
    ["Highest avg risk country", top_risk_country],
    ["Highest PM2.5 exceedance", highest_exceed_country],
], columns=["Metric", "Value"])

print(report.to_string(index=False))

---
## Export to HTML / PDF

Run from the repo root after executing this notebook:

```bash
# HTML (fast, self-contained, embeds all images)
jupyter nbconvert docs/pipeline_report.ipynb --to html --output docs/pipeline_report.html

# PDF (requires nbconvert + latex or webpdf backend)
jupyter nbconvert docs/pipeline_report.ipynb --to pdf --output docs/pipeline_report.pdf
# or via webpdf (no LaTeX needed, uses chromium):
jupyter nbconvert docs/pipeline_report.ipynb --to webpdf --output docs/pipeline_report.pdf
```